<!-- NOTEBOOK_METADATA source: "⚠️ Jupyter Notebook" title: "Observability for Google Gemini JS/TS with Langfuse" sidebarTitle: "Google Gemini (JS/TS)" logo: "/images/integrations/google_gemini_icon.svg" description: "Learn how to integrate Langfuse with the Google Gemini JS/TS SDK for tracing text generation, including the flushAsync pattern for serverless and Next.js environments." category: "Integrations" -->

# Trace Google Gemini with Langfuse (JS/TS)

<a href="https://langfuse.com/integrations/model-providers/google-gemini"><img className="inline" alt="Python" src="https://img.shields.io/badge/Python-3776AB?style=flat&logo=python&logoColor=white" /></a> <a href="https://langfuse.com/integrations/model-providers/google-gemini-js"><img className="inline" alt="JS/TS" src="https://img.shields.io/badge/JS/TS-d4d4d8?style=flat&logo=javascript&logoColor=white" /></a>

This cookbook shows how to trace [Google Gemini](https://ai.google.dev/gemini-api/docs) calls in JavaScript/TypeScript using the Langfuse SDK. Because there is no automatic OpenTelemetry instrumentation for the Gemini JS SDK yet, this guide uses the [Langfuse JS/TS SDK](https://langfuse.com/docs/observability/sdk/typescript/overview) to create traces and generations manually.

> **What is Google Gemini?** [Google Gemini](https://ai.google.dev/gemini-api/docs/libraries) is Google's family of multimodal generative models available through the Gemini API, with models like `gemini-2.0-flash` for fast, cost-efficient inference.

> **What is Langfuse?** [Langfuse](https://langfuse.com) is an open source LLM observability platform. It traces your AI calls and captures inputs, outputs, token usage, latency, and cost so you can debug and monitor your application.

<!-- STEPS_START -->
## Step 1: Install Dependencies

```bash
npm install @google/generative-ai langfuse
```

> **Note**: This cookbook uses **Deno.js** for execution, which requires different syntax for importing packages and setting environment variables. For Node.js applications, the setup is the same but use standard `npm` packages and `process.env`.

## Step 2: Configure Environment

Set your Langfuse API keys and your Google Gemini API key. Get Langfuse keys from [Langfuse Cloud](https://langfuse.com/cloud) or a [self-hosted instance](https://langfuse.com/self-hosting). Get your Gemini API key from [Google AI Studio](https://aistudio.google.com/app/apikey).

In [ ]:
// Langfuse authentication keys
Deno.env.set("LANGFUSE_PUBLIC_KEY", "pk-lf-...");
Deno.env.set("LANGFUSE_SECRET_KEY", "sk-lf-...");

// Langfuse host — 🇪🇺 EU region
// Other regions: 🇺🇸 US: https://us.cloud.langfuse.com, 🇯🇵 Japan: https://jp.cloud.langfuse.com, ⚕️ HIPAA: https://hipaa.cloud.langfuse.com
Deno.env.set("LANGFUSE_BASE_URL", "https://cloud.langfuse.com");

// Google Gemini API key
Deno.env.set("GEMINI_API_KEY", "...");

## Step 3: Initialize Langfuse

Create a single `Langfuse` client instance. In production applications (Express, Next.js, etc.) initialise this once at module level and reuse it across requests.

In [ ]:
import { Langfuse } from "npm:langfuse";

const langfuse = new Langfuse({
  publicKey: Deno.env.get("LANGFUSE_PUBLIC_KEY"),
  secretKey: Deno.env.get("LANGFUSE_SECRET_KEY"),
  baseUrl: Deno.env.get("LANGFUSE_BASE_URL"),
});

## Step 4: Trace a Text Generation

Wrap each Gemini call in a Langfuse **trace** (the top-level container) and a **generation** (the LLM span). Pass `usageMetadata` from the Gemini response to enable token-based cost tracking in Langfuse.

> **Model name for cost tracking**: Use the exact model string you pass to Gemini (e.g. `"gemini-2.0-flash"`) as the `model` field in the generation. Langfuse maps this name to its pricing registry to calculate cost automatically.

In [ ]:
import { GoogleGenerativeAI } from "npm:@google/generative-ai";
import { Langfuse } from "npm:langfuse";

const genAI = new GoogleGenerativeAI(Deno.env.get("GEMINI_API_KEY") ?? "");

const MODEL = "gemini-2.0-flash";
const prompt = "Explain what LLM observability is in two sentences.";

// Create a trace to group all operations for this request
const trace = langfuse.trace({
  name: "gemini-text-generation",
  input: { prompt },
});

// Create a generation span — opened before the API call
const generation = trace.generation({
  name: "gemini-generate-content",
  model: MODEL,
  input: [{ role: "user", content: prompt }],
});

const model = genAI.getGenerativeModel({ model: MODEL });
const result = await model.generateContent(prompt);

const text = result.response.text();
const usage = result.response.usageMetadata;

// Close the generation with the output and token counts
generation.end({
  output: text,
  usage: {
    input: usage?.promptTokenCount,
    output: usage?.candidatesTokenCount,
    total: usage?.totalTokenCount,
  },
});

trace.update({ output: text });

console.log(text);

// Flush all pending events to Langfuse before the process exits
await langfuse.flushAsync();

### View the Trace in Langfuse

After running the cell above, open your [Langfuse Trace Table](https://cloud.langfuse.com) to see the trace with input, output, token counts, and estimated cost.

## Step 5: Streaming

For streaming responses, open the generation before the stream starts, collect chunks, then close the generation once the stream finishes and `usageMetadata` is available.

In [ ]:
import { GoogleGenerativeAI } from "npm:@google/generative-ai";
import { Langfuse } from "npm:langfuse";

const genAI = new GoogleGenerativeAI(Deno.env.get("GEMINI_API_KEY") ?? "");

const MODEL = "gemini-2.0-flash";
const prompt = "List three benefits of LLM tracing.";

const trace = langfuse.trace({ name: "gemini-streaming", input: { prompt } });
const generation = trace.generation({
  name: "gemini-stream",
  model: MODEL,
  input: [{ role: "user", content: prompt }],
});

const model = genAI.getGenerativeModel({ model: MODEL });
const streamResult = await model.generateContentStream(prompt);

let fullText = "";
for await (const chunk of streamResult.stream) {
  const part = chunk.text();
  process.stdout.write(part);
  fullText += part;
}
console.log();

// usageMetadata is only available after the stream completes
const response = await streamResult.response;
const usage = response.usageMetadata;

generation.end({
  output: fullText,
  usage: {
    input: usage?.promptTokenCount,
    output: usage?.candidatesTokenCount,
    total: usage?.totalTokenCount,
  },
});

trace.update({ output: fullText });
await langfuse.flushAsync();

## The `flushAsync()` Pattern for Serverless and Next.js

Langfuse batches events and sends them in the background. In **serverless environments** (Next.js Route Handlers, Vercel Edge Functions, AWS Lambda), the process may be frozen or terminated before the background queue drains.

Always `await langfuse.flushAsync()` at the end of each handler to guarantee all events are delivered before the runtime is paused:

```ts
// app/api/chat/route.ts (Next.js App Router)
import { NextRequest, NextResponse } from "next/server";
import { GoogleGenerativeAI } from "@google/generative-ai";
import { Langfuse } from "langfuse";

const langfuse = new Langfuse({
  publicKey: process.env.LANGFUSE_PUBLIC_KEY!,
  secretKey: process.env.LANGFUSE_SECRET_KEY!,
  baseUrl: process.env.LANGFUSE_BASE_URL,
});

const genAI = new GoogleGenerativeAI(process.env.GEMINI_API_KEY!);

export async function POST(req: NextRequest) {
  const { prompt } = await req.json();

  const trace = langfuse.trace({ name: "chat", input: { prompt } });
  const generation = trace.generation({
    name: "gemini-generate",
    model: "gemini-2.0-flash",
    input: [{ role: "user", content: prompt }],
  });

  const model = genAI.getGenerativeModel({ model: "gemini-2.0-flash" });
  const result = await model.generateContent(prompt);
  const text = result.response.text();
  const usage = result.response.usageMetadata;

  generation.end({
    output: text,
    usage: {
      input: usage?.promptTokenCount,
      output: usage?.candidatesTokenCount,
      total: usage?.totalTokenCount,
    },
  });
  trace.update({ output: text });

  // Required in serverless — ensures events are sent before the function exits
  await langfuse.flushAsync();

  return NextResponse.json({ text });
}
```

<!-- STEPS_END -->

<!-- MARKDOWN_COMPONENT name: "LearnMore" path: "@/components-mdx/integration-learn-more-js.mdx" -->